# Introduction to LangChain

**Author:** Shinin Varongchayakul

**Date:** 30 Jan 2026

## 1. API Key

In [1]:
# Import packages
from pathlib import Path
from dotenv import load_dotenv
import os

# Get .env file path
PROJECT_ROOT = Path.cwd().parents[2]
env_path = PROJECT_ROOT / ".env"

# Load variables from .env
load_dotenv(env_path, override=True)

# Get Gemini API key
gemini_api_key = os.getenv("GEMINI_API_KEY_01")

## 2. LLM

In [2]:
# Import package
from langchain_google_genai import ChatGoogleGenerativeAI

# Create model instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5,
    api_key=gemini_api_key
)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## 3. Prompt Template

In [3]:
# Import package
from langchain_core.prompts import ChatPromptTemplate

# Define system prompt
system_prompt = """
You are an expert curator of mental models across science, philosophy, and applied reasoning.

Your task is to explain mental models clearly and accurately using a fixed schema.

If the origin of a model is unclear or debated, state that explicitly.

Do not invent historical sources. Be concise and concrete.
"""

# Define user prompt
user_prompt = "Explain the following mental model: {model_query}"

# Create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        # System prompt
        ("system", system_prompt.strip()),

        # User prompt
        ("human", user_prompt)
    ]
)

In [4]:
# Inspect prompt
prompt.format_messages(model_query="Pareto Principle")

[SystemMessage(content='You are an expert curator of mental models across science, philosophy, and applied reasoning.\n\nYour task is to explain mental models clearly and accurately using a fixed schema.\n\nIf the origin of a model is unclear or debated, state that explicitly.\n\nDo not invent historical sources. Be concise and concrete.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain the following mental model: Pareto Principle', additional_kwargs={}, response_metadata={})]

## 4. Output Format

In [5]:
# Import packages
from pydantic import BaseModel, Field
from typing import List, Literal

# Define output structure
class MentalModel(BaseModel):

    # Mental model name
    model_name: str = Field(description="The commonly accepted name of the mental model")

    # Source/origin
    origin: str = Field(
        description="Where the model comes from (person, book, field, or cultural origin)"
    )

    # Brief description
    description: str = Field(
        description="A brief explanation of what the mental model is and why it matters"
    )

    # Examples
    examples: List[str] = Field(
        description="Concrete real-world examples illustrating the mental model"
    )

    # Tags
    tags: List[str] = Field(
        description="Short tags such as decision-making, systems thinking, learning, philosophy"
    )

In [6]:
# Add output structure to LLM
llm_with_structured_output = llm.with_structured_output(MentalModel)

## 5. Chain

In [7]:
# Build chain
chain = prompt | llm_with_structured_output

## 6. Single Run

In [8]:
# Run query
result = chain.invoke({"model_query": "Compound Interest"})

In [9]:
# Import package
import json

# Load result
result_dict = result.model_dump()

# Print
print(json.dumps(result_dict, indent=4))

{
    "model_name": "Compound Interest",
    "origin": "Mathematics and Finance. While often popularized by figures like Benjamin Franklin and Albert Einstein, the mathematical concept of interest on interest has been understood and applied for centuries in financial practices.",
    "description": "Compound interest is the addition of interest to the principal sum of a loan or deposit, or in other words, interest on interest. It is the result of reinvesting interest, rather than paying it out, so that interest in the next period is then earned on the principal sum plus previously accumulated interest. This leads to exponential growth over time, making it a powerful force in finance and applicable to many areas where small, consistent gains accumulate.",
    "examples": [
        "A savings account earning 5% interest annually, where the interest earned each year is added to the principal, causing the total balance to grow faster than if only the initial principal earned interest.",
  

## 7. Batch run

In [10]:
# Create list of mental models
mental_model_queries = [
    "First Principles Thinking",
    "Occam's Razor",
    "Confirmation Bias",
    "Map is Not the Territory",
    "42"
]

# Create batch inputs
batch_inputs = [{"model_query": query} for query in mental_model_queries]

# Run queries
results = chain.batch(batch_inputs)

# Instantiate collector
query_collector = [result.model_dump() for result in results]

In [11]:
# Instantiate counter
i = 1

# Loop through elements in collector
for result in query_collector:

    # Print result
    print(f"👉 Query {i}:")
    print(json.dumps(result, indent=4))
    print("\n")

    # Add 1 to counter
    i += 1

👉 Query 1:
{
    "model_name": "First Principles Thinking",
    "origin": "Ancient Greek philosophy, particularly Aristotle; popularized in modern business by Elon Musk.",
    "description": "First Principles Thinking involves breaking down complex problems or systems into their fundamental truths, the basic irreducible elements, rather than reasoning by analogy or relying on existing assumptions. It matters because it enables true innovation, allows for novel solutions, and helps avoid the pitfalls of conventional wisdom.",
    "examples": [
        "Elon Musk's approach to building rockets at SpaceX: instead of buying expensive rockets, he questioned the cost of raw materials (aluminum alloys, titanium, carbon fiber) and realized they were relatively cheap, leading to a new, more efficient manufacturing process.",
        "Designing a new type of chair by considering the fundamental needs of human posture, support, and material properties, rather than simply modifying existing chair 